# Entity Matching Hands-on

Make sure you have installed all the required packages by running `uv sync` in your terminal (see the README), and select the `.venv` as your notebook kernel.

Copy `.env.example` to `.env` and fill in your CDF client credentials before running the cell below.

**Connect Using Client secret**

In [ ]:
import pprint
import os

from dotenv import load_dotenv

from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials
from cognite.client.exceptions import CogniteAPIError, CogniteDuplicatedError, CogniteNotFoundError

pp = pprint.PrettyPrinter(indent=2)

load_dotenv(override=True)

credentials = OAuthClientCredentials(
    token_url=os.environ["COGNITE_TOKEN_URL"],
    client_id=os.environ["COGNITE_CLIENT_ID"],
    client_secret=os.environ["COGNITE_CLIENT_SECRET"],
    scopes=os.environ["COGNITE_TOKEN_SCOPES"].split(),
)
config = ClientConfig(
    client_name="my-special-client",
    project=os.environ["COGNITE_PROJECT"],
    base_url=os.environ["COGNITE_BASE_URL"],
    credentials=credentials,
)
client = CogniteClient(config)

In [ ]:
from cognite.client.data_classes import DataSetWrite

# Create datasets
try:
    bra_dataset = client.data_sets.create(DataSetWrite(external_id="BRA_TestData", name='TestDataset', write_protected=True))
except CogniteDuplicatedError as e:
    print("BRA_TestData already exists, you can continue")
    bra_dataset = client.data_sets.retrieve(external_id="BRA_TestData")
try:
    nor_dataset = client.data_sets.create(DataSetWrite(external_id="NOR_TestData", name='TestDataset', write_protected=True))
except CogniteDuplicatedError as e:
    print("NOR_TestData already exists, you can continue")
    nor_dataset = client.data_sets.retrieve(external_id="NOR_TestData")
assert nor_dataset is not None and bra_dataset is not None

## Create some resources to match

In [ ]:
data_foundation = [
    # Without CountryCode
  {"external_id" : "Main_Test_Platform_13", "parent_external_id": None, "description": "Main Test Platform 13", "name": "MainTestPlatform13"},
  {"external_id" : "Substation_Platform_13", "parent_external_id": "Main_Test_Platform_13", "description": "Substation of Main Test Platform 13", "name": "Substation_Platform13"},
  {"external_id" : "Pump1_Substation_Platform_13", "parent_external_id": "Substation_Platform_13", "description": "Pump 1 - exists on Substation Platform 13", "name": "Pump1_Substation_Platform13"},
  {"external_id" : "Pump2_Substation_Platform_13", "parent_external_id": None, "description": "Pump 2 - exists on Substation Platform 13", "name": "Pump2_SubstationPlatform13"},
    # Outliers: Different Pattern
  {"external_id" : "MainTestPlatform13", "parent_external_id": None, "description": "Main Test Platform 13", "name": "MainTestPlatform13"},
  {"external_id" : "Platform13", "parent_external_id": None, "description": "Main Test Platform 13", "name": "MainTestPlatform13"},
    # Outliers: Description
  {"external_id" : "MeinTestPlatform13", "parent_external_id": None, "description": "Main Test Platform 13", "name": "MainTestPlatform13"},
    # Norwegian assets
  {"external_id" : "NOR_Main_Test_Platform_13", "parent_external_id": None, "description": "Main Test Platform. Number 13. Located in Norway", "name": "Main Test Platform 13", "dataset": "NOR_TestData"},
  {"external_id" : "NOR_Substation_Platform_13", "parent_external_id": "NOR_Main_Test_Platform_13", "description": "Substation of Main Test Platform 13", "name": "Substation Platform 13", "dataset": "NOR_TestData"},
  {"external_id" : "NOR_Pump1_Substation_Platform_13", "parent_external_id": "NOR_Substation_Platform_13", "description": "Pump 1 - exists on Substation Platform 13", "name": "Pump1 Substation Platform 13", "dataset": "NOR_TestData"},
  {"external_id" : "NOR_Pump2_SubstationPlatform_13", "parent_external_id": None, "description": "Pump 2 - exists on Substation Platform 13", "name": "Pump2 Substation Platform 13", "dataset": "NOR_TestData"},
    # Brazilian assets
  {"external_id" : "BRA_Main_Test_Platform_13", "parent_external_id": None, "description": "Main Test Platform. Number 13. Located in Brazil", "name": "Main Test Platform 13", "dataset": "BRA_TestData"},
  {"external_id" : "BRA_Substation_Platform_13", "parent_external_id": "BRA_Main_Test_Platform_13", "description": "Substation of Main Test Platform 13", "name": "Substation Platform 13", "dataset": "BRA_TestData"},
  {"external_id" : "BRA_Pump1_Substation_Platform_13", "parent_external_id": "BRA_Substation_Platform_13", "description": "Pump 1 - exists on Substation Platform 13", "name": "Pump1 Substation Platform 13", "dataset": "BRA_TestData"},
  {"external_id" : "BRA_Pump2_SubstationPlatform_13", "parent_external_id": None, "description": "Pump 2 - exists on Substation Platform 13", "name": "Pump2 Substation Platform 13", "dataset": "BRA_TestData"},
    # Create outsider that should be matched on description
  {"external_id" : "x", "parent_external_id": None, "description": "Measurements on the Main Test Platform 13. Located in Norway", "name": "x", "dataset": "NOR_TestData"},

]

# RESET
# for _asset in data_foundation:
#     try:
#         client.assets.delete(external_id=_asset.get("external_id"),recursive=True)
#     except CogniteNotFoundError:
#         pass
#     try:
#         client.time_series.delete(external_id=_asset.get("external_id")+"_temperature")
#         client.time_series.delete(external_id=_asset.get("external_id")+"_humidity")
#     except CogniteNotFoundError:
#         pass

# time.sleep(5)

In [ ]:
# Create assets
from cognite.client.data_classes import AssetWrite

assets_external_ids=[]
for _asset in data_foundation[:-1]:
    if _asset.get("dataset")=="NOR_TestData":
        dataset_id = nor_dataset.id
    elif _asset.get("dataset") == "BRA_TestData":
        dataset_id = bra_dataset.id
    else:
        dataset_id = None
    try:
        client.assets.create(asset=AssetWrite(external_id=_asset.get("external_id"), name=_asset.get("name"), description=_asset.get("description"), data_set_id=dataset_id, parent_external_id=_asset.get("parent_external_id"))) # type: ignore
    except CogniteDuplicatedError:
        pass
    assets_external_ids.append(_asset.get('external_id'))

In [ ]:
# Create timeseries to match
from cognite.client.data_classes import TimeSeriesWrite

ts_external_ids=[]
for _asset in data_foundation:
    if _asset.get("dataset")=="NOR_TestData":
        dataset_id = nor_dataset.id
    elif _asset.get("dataset") == "BRA_TestData":
        dataset_id = bra_dataset.id
    else:
        dataset_id=None
    try:
        client.time_series.create(TimeSeriesWrite(external_id=_asset.get('external_id')+"_temperature", name=_asset.get('external_id')+"_temperature", description="Temperature."+_asset.get("description"), data_set_id=dataset_id)) # type: ignore
    except CogniteDuplicatedError:
        pass
    try:
        client.time_series.create(TimeSeriesWrite(external_id=_asset.get('external_id')+"_humidity", name=_asset.get('external_id')+"_humidity", description="Humidity."+_asset.get("description"), data_set_id=dataset_id))  # type: ignore
    except CogniteDuplicatedError:
        pass
    ts_external_ids.append(_asset.get('external_id')+'_temperature')
    ts_external_ids.append(_asset.get('external_id')+'_humidity')

try:
    client.time_series.create(TimeSeriesWrite(external_id="MMMM_TestPlattform_13_temperature", name="MMMM_TestPlattform_13_temperature", description="Temperature that belongs to MainTestPlatform_13. Located in Norway.", data_set_id=nor_dataset.id)) # type: ignore
except CogniteDuplicatedError:
    pass
ts_external_ids.append("MMMM_TestPlattform_13_temperature")

In [ ]:
# Creation done

print(
    "Assets:\n", assets_external_ids
)
print(
    "Time Series:\n ", ts_external_ids
)

In [ ]:
print("=====================================")
print("Let's prepare to match")
print("=====================================")

sources = []
print("Sources:")
_timeseries = client.time_series.retrieve_multiple(external_ids=ts_external_ids)
sources = [{"id" : ts.id, "name" : ts.name, "description": ts.description, "externalId": ts.external_id} for ts in _timeseries]

for x in range(len(sources)):
  print(sources[x])

print("=====================================")

_assets = client.assets.retrieve_multiple(external_ids=assets_external_ids)
print("Targets:")
targets = [{"id" : asset.id, "name" : asset.name, "description": asset.description, "externalId": asset.external_id} for asset in _assets]

for x in range(len(targets)):
  print(targets[x])

In [ ]:
# 1. Fit a supervised model using "name"
true_matches = [{"sourceExternalId":"Main_Test_Platform_13_humidity", "targetExternalId": "Main_Test_Platform_13"},
        {"sourceExternalId":"Substation_Platform_13_humidity", "targetExternalId": "Substation_Platform_13"},
        {"sourceExternalId":"Pump1_Substation_Platform_13_humidity", "targetExternalId": "Pump1_Substation_Platform_13"},]

sup1_model = client.entity_matching.fit(
                sources = sources,
                targets = targets,
                true_matches = true_matches,
                match_fields = [("name", "name")],
                description="First")

print(sup1_model.to_pandas())

# Predict using the entity matching model
job = sup1_model.predict(num_matches = 4)
matches = job.get_result()
pp.pprint(matches["items"])

In [ ]:
# 2. Matchfields being external id instead
sup2_model = client.entity_matching.fit(
                sources = sources,
                targets = targets,
                true_matches = true_matches,
                match_fields = [("externalId", "externalId")],
                description="Second")

print(sup2_model.to_pandas())
job = sup2_model.predict(num_matches = 4)
matches = job.get_result()
pp.pprint(matches["items"])


In [ ]:
# 3. ADDING more matches and refitting existing model
new_true_matches = [*true_matches,
        {"sourceExternalId":"NOR_Main_Test_Platform_13_humidity", "targetExternalId": "NOR_Main_Test_Platform_13"},
        {"sourceExternalId":"NOR_Substation_Platform_13_humidity", "targetExternalId": "NOR_Substation_Platform_13"},
        {"sourceExternalId":"NOR_Pump1_Substation_Platform_13_humidity", "targetExternalId": "NOR_Pump1_Substation_Platform_13"},

        {"sourceExternalId":"BRA_Main_Test_Platform_13_humidity", "targetExternalId": "BRA_Main_Test_Platform_13"},
        {"sourceExternalId":"BRA_Substation_Platform_13_humidity", "targetExternalId": "BRA_Substation_Platform_13"},
        {"sourceExternalId":"BRA_Pump1_Substation_Platform_13_humidity", "targetExternalId": "BRA_Pump1_Substation_Platform_13"},]

sup3_model = sup1_model.refit(true_matches=new_true_matches)

print(sup3_model.to_pandas())

# Predict using the entity matching model
len_targets = len(targets)
job = sup3_model.predict(num_matches = len_targets)

matches = job.get_result()

pp.pprint(matches["items"])

In [ ]:
# 4. Running on X
sup4_model = client.entity_matching.fit(
                sources = sources,
                targets = targets,
                true_matches = new_true_matches,
                match_fields = [("description", "description")],
                description="Fourth")

sup4_model.to_pandas()
job = sup4_model.predict(num_matches = 4)
matches = job.get_result()
pp.pprint(matches["items"])


In [ ]:
# Try to use bigram instead
sup5_model = client.entity_matching.fit(
                sources = sources,
                targets = targets,
                true_matches = new_true_matches,
                feature_type="bigram",
                match_fields = [("description", "description")], # refers to the strings in
                description="Second Attempt X")

print(sup5_model.to_pandas())
job = sup5_model.predict(num_matches = 4)
matches = job.get_result()
pp.pprint(matches["items"])

In [ ]:
# Filtering on dataset
assets_in_dataset=client.assets.list(data_set_external_ids="NOR_TestData")

# Filtering using retrieve_subtree
relevant_asset_subtree = client.assets.retrieve_subtree(external_id="NOR_Main_Test_Platform_13", depth=5)

print(f"Asset subtree for NOR_Main_Test_Platform_13: {relevant_asset_subtree.to_pandas()}")

# Get timeseries that aren't already connected to assets
# Scope down to dataset
ts = client.time_series.list(data_set_external_ids=nor_dataset.external_id)
# Scope down to timeseries that are not connected to assets
filtered_ts = [t for t in ts if t.asset_id is None]
print(f"Time series that are not connected to assets: {[t.external_id for t in filtered_ts]}")



sources = []
print("Sources:")
sources = [{"id" : ts.id, "name" : ts.name, "description": ts.description} for ts in filtered_ts]

for x in range(len(sources)):
  print(sources[x])

print("=====================================")

assets = relevant_asset_subtree
print("Targets:")
targets = [{"id" : asset.id, "name" : asset.name, "description": asset.description} for asset in assets]

for x in range(len(targets)):
  print(targets[x])

sup_model = client.entity_matching.fit(
                sources = sources,
                targets = targets,
                true_matches = true_matches,
                feature_type="bigram",
                match_fields = [("externalId", "externalId")], # refers to the strings in
                description="EntityMatchingJob1337")

sup_model.to_pandas()
job = sup_model.predict(num_matches = 4)
matches = job.get_result()
pp.pprint(matches["items"])


In [ ]:
# Fit a un-supervised model
unsupervised_model = client.entity_matching.fit(sources = sources,
                                   targets = targets)

unsupervised_model.to_pandas()

In [ ]:
#Predict using an entity matching model
job = unsupervised_model.predict(num_matches = 6)
matches = job.get_result()
pp.pprint(matches["items"])


In [ ]:
# Fit a un-supervised model
unsupervised_model = client.entity_matching.fit(sources = sources,
                                   targets = targets,
                                    match_fields=[("externalId","externalId")])

unsupervised_model.to_pandas()

In [ ]:
#Predict using an entity matching model
job = unsupervised_model.predict(num_matches = 6)
matches = job.get_result()
pp.pprint(matches["items"])


In [ ]:
# Fit a un-supervised model
unsupervised_model = client.entity_matching.fit(sources = sources,
                                   targets = targets,
                                match_fields=[("externalId","externalId"),("description","description")],
                                feature_type="bigram")

unsupervised_model.to_pandas()

In [ ]:
#Predict using an entity matching model
job = unsupervised_model.predict(num_matches = 6)
matches = job.get_result()
pp.pprint(matches["items"])

## Add additional key match_fields
When to use the key match_fields?
By default only name in sources and name in targets are used to calculate similarity measures. The match_fields parameter lets you specify all combinations of fields in sources and targets that should be used to calculate features.

In this example it looks like also comparing the description field for both targets and sources will improve the model.

Note: Calculating similarity measures can be time consuming. Therefore, we avoid adding match_fields combinations which adds little or no information to the model.

In [ ]:
try:
    model = client.entity_matching.fit(sources = sources,
                                       targets = targets,
                                       match_fields = [("name", "name"), ("description", "description")]
                                      )
except CogniteAPIError as error:
    print(error)


## To not get not get an error if some items in sources or targets having missing values add `ignore_missing_fields=True`.

In [ ]:
model = client.entity_matching.fit(sources = sources,
                                   targets = targets,
                                   match_fields = [("name", "name"), ("description", "description")],
                                   ignore_missing_fields = True
)

In [ ]:

# Predict on training data
job = model.predict(num_matches = 2)
matches = job.get_result()
pp.pprint(matches["items"])